In [ ]:
# ============================================================
# GOOGLE COLAB SETUP — run this cell first when using Colab
# ============================================================
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # 1. Mount Google Drive so your data is accessible
    from google.colab import drive
    drive.mount('/content/drive')

    # 2. Set REPO_PATH to wherever you stored (or will store) the repo on Drive.
    #    If the folder doesn't exist the repo is cloned there automatically.
    REPO_PATH = '/content/drive/MyDrive/Factor-Research'

    if not os.path.exists(REPO_PATH):
        print('Cloning repository to Google Drive...')
        os.system(f'git clone https://github.com/mbrennan5/Factor-Research.git {REPO_PATH}')
    else:
        print(f'Repository found at {REPO_PATH}')

    # 3. Install required packages.
    #    Most are already in Colab; only the non-standard ones need installing.
    print('Installing packages...')
    os.system('pip install -q lightgbm xgboost optuna plotly tqdm yfinance')

    # 4. Change to the notebooks directory so relative paths (../data/...) work.
    NOTEBOOKS_DIR = os.path.join(REPO_PATH, 'notebooks')
    os.chdir(NOTEBOOKS_DIR)
    print(f'Working directory set to: {os.getcwd()}')
else:
    print('Running locally — no Colab setup needed.')


# Data Preparation and Feature Engineering

This notebook is the **first step** in the quantitative research pipeline. It handles the loading, cleaning, and preprocessing of raw minute-level stock data and Barra factor data.

## **Objective**

1.  **Load Raw Data**: Ingest minute-level OHLCV (Open, High, Low, Close, Volume) data and Barra factor data.
2.  **Data Cleaning**: Handle missing values, sort data by time, and ensure data integrity.
3.  **Feature Engineering**: Calculate key financial metrics that will be used as base features for alpha factor generation.
4.  **Save Processed Data**: Store the cleaned and engineered data in a structured format for downstream use.

## **Key Features Generated**

-   **Daily Return Rate**: Core metric for stock performance.
-   **Daily Turnover Rate**: Liquidity indicator.
-   **VWAP (Volume-Weighted Average Price)**: Price metric adjusted for trading volume.
-   **Barra Size Factor**: Market capitalization-based risk factor.


In [ ]:
# === 1. Import Libraries and Load Minute-Level Stock Data ===

import pandas as pd
import numpy as np
import glob
from tqdm import tqdm
import warnings
import os

warnings.filterwarnings('ignore')

# --- Define Data Loading Function ---
def load_and_prepare_data(path, file_limit=None):
    """
    Loads all CSV files from a given path, concatenates them,
    and performs initial cleaning and type conversion.

    Args:
        path (str): The directory path containing the CSV files.
        file_limit (int, optional): The maximum number of files to load. Defaults to None (all files).

    Returns:
        pandas.DataFrame: A single DataFrame containing all loaded and prepared data,
                          or None if no files were found.
    """
    print(f"--- Loading data from: {path} ---")
    all_files = glob.glob(os.path.join(path, "*.csv"))
    
    if file_limit:
        all_files = all_files[:file_limit]
        print(f"Applied file limit: Loading {len(all_files)} files.")

    if not all_files:
        print(f"⚠️ Warning: No CSV files found in '{path}'.")
        return None

    li = []
    for filename in tqdm(all_files, desc=f"Processing files in {os.path.basename(path)}"):
        try:
            df = pd.read_csv(filename, index_col=None, header=0)
            li.append(df)
        except Exception as e:
            print(f"Error loading {filename}: {e}")
            
    if not li:
        print("⚠️ Warning: No data was loaded.")
        return None

    frame = pd.concat(li, axis=0, ignore_index=True)
    print(f"✅ Successfully loaded and concatenated {len(li)} files.")
    
    # --- Data Cleaning and Type Conversion ---
    # Convert 'datetime' to datetime objects for proper sorting
    frame['datetime'] = pd.to_datetime(frame['datetime'])
    
    # Ensure numerical columns are numeric, coercing errors
    for col in ['open', 'high', 'low', 'close', 'volume', 'money']:
        frame[col] = pd.to_numeric(frame[col], errors='coerce')
        
    # Drop rows with critical missing values after coercion
    frame.dropna(subset=['datetime', 'open', 'close', 'volume'], inplace=True)
    
    return frame

# --- Load Data from Both Sources ---
# Define paths relative to the project root
path1 = '../data/raw/datamin2'
path2 = '../data/raw/datamin3'

# Load the dataframes
df1 = load_and_prepare_data(path1)
df2 = load_and_prepare_data(path2)


In [ ]:
# === 2. Combine, Sort, and Finalize Initial DataFrame ===

# --- Combine DataFrames ---
# Check if both DataFrames were loaded successfully before combining
if df1 is not None and df2 is not None:
    result_df = pd.concat([df1, df2], ignore_index=True)
    print("--- Combining DataFrames ---")
    print("✅ df1 and df2 successfully combined.")
elif df1 is not None:
    result_df = df1
    print("--- Using df1 only ---")
elif df2 is not None:
    result_df = df2
    print("--- Using df2 only ---")
else:
    result_df = None
    print("❌ Error: No data was loaded from any source. Cannot proceed.")

if result_df is not None:
    # --- Sort Values ---
    # Sort by stock ID and then by datetime to ensure correct chronological order for time-series calculations
    print("\n--- Sorting DataFrame ---")
    result_df.sort_values(by=['order_book_id', 'datetime'], inplace=True)
    print("✅ DataFrame sorted by 'order_book_id' and 'datetime'.")

    # --- Reset Index ---
    # Reset the index to be clean and sequential after sorting
    result_df.reset_index(drop=True, inplace=True)
    print("✅ Index has been reset.")

    # --- Add Date Column ---
    # Extract the date part from the datetime for daily aggregations
    result_df['date'] = result_df['datetime'].dt.date
    print("✅ 'date' column created.")

    # --- Display Final DataFrame Info ---
    print("\n--- Final DataFrame Summary ---")
    print(f"Total records: {len(result_df):,}")
    print(f"Unique stocks: {result_df['order_book_id'].nunique()}")
    print(f"Date range: {result_df['date'].min()} to {result_df['date'].max()}")
    print("\nDataFrame head:")
    print(result_df.head())


In [ ]:
# === 3. Feature Engineering: Daily Return Rate ===

if result_df is not None:
    print("--- Calculating Daily Return Rate ---")
    
    # --- Get Daily Closing Prices ---
    # Group by stock and date, then take the last known price of the day as the daily close.
    daily_close = result_df.groupby(['order_book_id', 'date'])['close'].last().reset_index()
    daily_close.rename(columns={'close': 'daily_close'}, inplace=True)
    print("✅ Calculated daily closing prices.")

    # --- Pivot to Wide Format ---
    # Reshape the data so that rows are dates and columns are stock IDs.
    # This format is essential for most quantitative analysis.
    daily_close_wide = daily_close.pivot(index='date', columns='order_book_id', values='daily_close')
    print("✅ Pivoted daily close data to wide format (dates x stocks).")

    # --- Calculate Daily Percentage Change (Return) ---
    # The pct_change() function calculates the return from the previous day's close.
    # fill_method='ffill' is used to forward-fill missing values before calculation to avoid excessive NaNs.
    daily_ret = daily_close_wide.pct_change().fillna(0)
    print("✅ Calculated daily percentage returns.")

    # --- Display Result ---
    print("\n--- Daily Return Data (Head) ---")
    print(daily_ret.head())
    
else:
    print("❌ Cannot calculate daily return rate because the base DataFrame is not available.")


In [ ]:
# === 4. Feature Engineering: Daily Turnover Rate ===

if result_df is not None:
    print("--- Calculating Daily Turnover Rate ---")
    
    # --- Aggregate Daily Volume and Money ---
    # Group by stock and date, then sum the volume and money traded for each day.
    daily_agg = result_df.groupby(['order_book_id', 'date']).agg(
        total_volume=('volume', 'sum'),
        total_money=('money', 'sum')
    ).reset_index()
    print("✅ Aggregated daily total volume and money.")

    # --- Pivot to Wide Format ---
    # Create separate wide-format DataFrames for daily volume and money.
    daily_volume_wide = daily_agg.pivot(index='date', columns='order_book_id', values='total_volume')
    daily_money_wide = daily_agg.pivot(index='date', columns='order_book_id', values='total_money')
    print("✅ Pivoted daily volume and money data to wide format.")

    # --- Calculate Daily Turnover Rate ---
    # Turnover rate is calculated as: Daily Volume / (Previous Day's Volume)
    # This measures the change in trading activity.
    daily_turnover = (daily_volume_wide / daily_volume_wide.shift(1)).fillna(0)
    
    # Replace infinite values that can occur from division by zero with NaN, then fill with 0
    daily_turnover.replace([np.inf, -np.inf], np.nan, inplace=True)
    daily_turnover.fillna(0, inplace=True)
    print("✅ Calculated daily turnover rate.")
    
    # --- Display Result ---
    print("\n--- Daily Turnover Rate Data (Head) ---")
    print(daily_turnover.head())
    
else:
    print("❌ Cannot calculate turnover rate because the base DataFrame is not available.")


In [ ]:
# === 5. Feature Engineering: Daily VWAP ===

if result_df is not None:
    print("--- Calculating Daily VWAP (Volume-Weighted Average Price) ---")
    
    # --- Calculate VWAP ---
    # VWAP is a more representative daily price than the simple close price.
    # It's calculated as: Sum(Close Price * Volume) / Sum(Volume) for each day.
    
    # First, calculate 'close' * 'volume' for each minute record.
    result_df['close_vol'] = result_df['close'] * result_df['volume']
    
    # Then, group by stock and date and sum the 'close_vol' and 'volume'.
    vwap_cal = result_df.groupby(['order_book_id', 'date']).agg(
        total_close_vol=('close_vol', 'sum'),
        total_volume=('volume', 'sum')
    ).reset_index()
    
    # Finally, compute VWAP.
    vwap_cal['vwap'] = vwap_cal['total_close_vol'] / vwap_cal['total_volume']
    print("✅ Calculated daily VWAP values.")

    # --- Pivot to Wide Format ---
    vwap_wide = vwap_cal.pivot(index='date', columns='order_book_id', values='vwap').fillna(0)
    print("✅ Pivoted VWAP data to wide format.")
    
    # --- Calculate VWAP Daily Return ---
    # Calculate the daily percentage change of the VWAP.
    vwap1pct = vwap_wide.pct_change().fillna(0)
    print("✅ Calculated daily percentage returns based on VWAP.")

    # --- Display Result ---
    print("\n--- Daily VWAP Return Data (Head) ---")
    print(vwap1pct.head())

else:
    print("❌ Cannot calculate VWAP because the base DataFrame is not available.")


In [ ]:
# === 6. Barra Factor Integration: Size Factor ===

# --- Define Barra Data Loading Function ---
def load_barra_data(path, date_index):
    """
    Loads all Barra factor files, processes them, and aligns them to the project's date index.
    
    Args:
        path (str): The directory path for Barra data.
        date_index (pd.DatetimeIndex): The target date index for alignment.

    Returns:
        pandas.DataFrame: A single DataFrame with the Barra 'size' factor,
                          or None if loading fails.
    """
    print(f"--- Loading Barra Data from: {path} ---")
    all_files = glob.glob(os.path.join(path, "*.csv"))

    # Filter out temporary or metadata files like '._*'
    all_files = [f for f in all_files if not os.path.basename(f).startswith('._')]

    if not all_files:
        print(f"⚠️ Warning: No valid Barra CSV files found in '{path}'.")
        return None

    li = []
    for filename in tqdm(all_files, desc="Processing Barra files"):
        try:
            df = pd.read_csv(filename, index_col=0)
            if not df.empty:
                # Extract date from the filename (e.g., 'data_barra_2023-10-31.csv')
                date_str = os.path.basename(filename).split('_')[-1].replace('.csv', '')
                df['date'] = pd.to_datetime(date_str)
                li.append(df)
        except Exception as e:
            print(f"Error loading {filename}: {e}")

    if not li:
        print("⚠️ Warning: No Barra data was loaded.")
        return None
        
    barra_df = pd.concat(li, axis=0, ignore_index=True)
    print(f"✅ Successfully loaded and combined {len(li)} Barra files.")
    
    # --- Process and Pivot Size Factor ---
    # Select only the 'size' factor as required
    size_df = barra_df[['date', 'order_book_id', 'size']]
    
    # Pivot to wide format
    size_wide = size_df.pivot(index='date', columns='order_book_id', values='size')
    print("✅ Pivoted Barra 'size' factor to wide format.")
    
    # --- Align with Project Date Index ---
    # Reindex the Barra data to match the main project's trading days.
    # This ensures consistency and handles any missing dates.
    aligned_size = size_wide.reindex(date_index).ffill()
    print("✅ Aligned Barra data with the project's date index.")
    
    return aligned_size

# --- Load and Process Barra Data ---
barra_path = '../data/raw/datamin4/data_barra'
if 'daily_ret' in locals() and daily_ret is not None:
    barra_size = load_barra_data(barra_path, daily_ret.index)
    if barra_size is not None:
        print("\n--- Barra Size Factor Data (Head) ---")
        print(barra_size.head())
else:
    print("❌ Cannot load Barra data because daily return data (for date index) is not available.")
    barra_size = None


In [ ]:
# === 7. Save All Processed Data ===

print("--- Saving all processed data to files ---")

# --- Create Output Directory ---
output_dir = '../data/processed/wide_data_preparation'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"✅ Created directory: {output_dir}")

# --- Define a saving function for consistency ---
def save_dataframe(df, name, directory):
    """Saves a DataFrame to a CSV file if it exists."""
    if df is not None:
        file_path = os.path.join(directory, f"{name}.csv")
        try:
            df.to_csv(file_path)
            print(f"✅ Successfully saved '{name}' to {file_path}")
        except Exception as e:
            print(f"❌ Error saving '{name}': {e}")
    else:
        print(f"⚠️ Skipping '{name}' because it was not generated.")

# --- Save all generated features ---
save_dataframe(daily_ret, 'daily_ret_data', output_dir)
save_dataframe(daily_turnover, 'daily_turnover_data', output_dir)
save_dataframe(daily_money_wide, 'daily_money_data', output_dir)
save_dataframe(vwap_wide, 'vwap_daily_data', output_dir)
save_dataframe(vwap1pct, 'vwap1pct_daily_data', output_dir)
save_dataframe(daily_close_wide, 'daily_close_data', output_dir)
save_dataframe(barra_size, 'barra_size_data', output_dir)

print("\n--- Data Preparation and Feature Engineering Complete ---")
